# Wheel Profile Shape and Empty-Car Hunting

I test whether tread-shape measurements add information beyond mean conicity
and rail profile. This is one fixed feature comparison using the same Gaussian
kernel ridge classifier and car-grouped validation as the contact experiment.


## Reproduction

We use this notebook for our profile-model comparisons in the report and presentation. Install Python 3.12 and the packages in [requirements.txt](../requirements.txt), following [INSTALL.md](../INSTALL.md). We need only the base packages; this notebook does not require TensorFlow, XGBoost or MATLAB.

Get the data through the private **-MxV-Wheel-Profile** course Teams channel: **Shared → Documents → Data**. Download `Profiles/`, `Eq_Con_Fit_Results.xlsm` and `final_merged_data.csv`, preserving the original folder structure. These profile experiments read:

```text
data/
├── final_merged_data.csv
└── Profiles/
    ├── Candidate wheel profiles/Car*_Axle*[LR].WHL
    └── WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv
```

We expect all 112 measured WHL files, separate from the unworn reference. We also accept `final_merged_data(in).csv`. If you keep the downloaded folder outside the repository, set its location before starting Jupyter:

macOS/Linux:
```bash
export HUNTING_DATA_DIR="/full/path/to/Data"
```

Windows PowerShell:
```powershell
$env:HUNTING_DATA_DIR = "C:\full\path\to\Data"
```

The default is the repository's `data/` folder. A locally rebuilt `outputs/data/final_merged_data.csv` takes precedence; regenerate or remove it when changing input releases.

From the repository root, activate the environment and start `python -m notebook`. Open this notebook and choose **Restart Kernel and Run All Cells**. We include the experiment functions below, so no unpublished helper modules, previous notebook session or historical-result files are required. The existing `src/contact_hunting.py` and `src/profile_shapes.py` supply the shared kernel and WHL readers.

We export figures as 300 dpi PNGs and vector PDFs and keep the tables behind them under the local output folder named below. The model runs recompute settings and predictions, using complete-car folds and 2,000 bootstrap draws with seed 42. To inspect a completed run without retraining, open its local tables or saved executed notebook. Those results apply to that run's inputs and code; rerun after either changes.

Keep partner-derived tables, images and executed notebooks in the private Teams storage. The Git notebook has cleared outputs. Confirm instructor access to the private data and any saved intermediates before submission.

**Output folder:** `outputs/profile_shape/`. The within-car scatter compares Truck 2 minus Truck 1 for both shape and displacement hunting count. The performance bars compare the two feature sets for acceleration and displacement separately.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_folder = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "src" / "contact_hunting.py").is_file()), None)
if project_folder is None:
    raise RuntimeError("Start Jupyter in the repository root or notebooks/; see INSTALL.md.")
sys.path.insert(0, str(project_folder / "src"))
configured_data = Path(os.environ.get("HUNTING_DATA_DIR", "data")).expanduser()
data_folder = configured_data if configured_data.is_absolute() else project_folder / configured_data

def merged_path(root):
    paths = [Path(root) / "outputs/data/final_merged_data.csv",
             data_folder / "final_merged_data.csv", data_folder / "final_merged_data(in).csv"]
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("Missing merged CSV. See this notebook's input instructions and INSTALL.md.")

from contact_hunting import (
    prepare_trucks, evaluate_models,
    paired_car_intervals, classification_metrics,
)
from profile_shapes import prepare_truck_shapes, SHAPE_MODELS, SHAPE_COMPARISONS

figure_folder = project_folder / "outputs" / "profile_shape"
figure_folder.mkdir(parents=True, exist_ok=True)
merged_data = pd.read_csv(merged_path(project_folder))
contact_data = pd.read_csv(data_folder / "Profiles/WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv", encoding="cp1252")
truck_data = prepare_trucks(merged_data, contact_data)
shape_data, wheel_data, profiles = prepare_truck_shapes(
    data_folder / "Profiles" / "Candidate wheel profiles", truck_data
)
truck_data = truck_data.merge(shape_data, on=["Car", "Truck"], validate="many_to_one")
display(pd.Series({
    "Wheel profiles": len(wheel_data),
    "Cars": truck_data.Car.nunique(),
    "Trucks": len(shape_data),
    "Truck/rail assessments": len(truck_data),
}, name="Coverage"))


## Shape measurements

The reader checks point counts, finite coordinates, increasing x positions in
the tread band, and coverage of x=60–110. It interpolates at one-unit spacing
and subtracts the height at x=85. All processing happens in memory.

Three measurements describe each truck:

- Mean fitted tread slope across its four wheels.
- Mean RMS departure from a straight line across its four wheels.
- Larger left/right RMS shape difference across its two wheelsets.

The window and measurements were fixed before this comparison. Both sides use
the export's existing x orientation. Units, alignment, and reconstruction
provenance remain provisional; these are exported coordinate descriptors,
not validated physical curvature or equivalent conicity.


## Evaluation

One complete car is held out at a time. Three grouped folds within the training
cars choose regularization and kernel width. Scaling is fitted on training
rows only. Both feature sets use the same six-setting grid and distance scale.

Acceleration is the main exploratory criterion; displacement is a separate
sensitivity check. A speed below 80 mph indicates hunting. For displacement,
the lower of the two axle speeds determines the truck label. A value of 80
means no hunting recorded within the tested range.


In [ ]:
target_names = {
    "acceleration_hunted": "Acceleration",
    "displacement_hunted": "Displacement sensitivity",
}
metric_tables = []
comparison_tables = []
local_predictions = {}

for target, label in target_names.items():
    metrics, predictions, settings = evaluate_models(truck_data, target, SHAPE_MODELS)
    intervals, comparisons = paired_car_intervals(
        predictions, n_bootstrap=2000, seed=42, comparisons=SHAPE_COMPARISONS
    )
    metrics["criterion"] = label
    comparisons["criterion"] = label
    metric_tables.append(metrics)
    comparison_tables.append(comparisons)
    local_predictions[target] = predictions

results = pd.concat(metric_tables, ignore_index=True)
comparisons = pd.concat(comparison_tables, ignore_index=True)
performance = results.set_index(["criterion", "model"])
display((100 * performance[["balanced_accuracy", "recall", "specificity"]]).round(1)
        .rename(columns={"balanced_accuracy": "Balanced accuracy (%)",
                         "recall": "Hunting recall (%)",
                         "specificity": "No-hunting recall (%)"}))


## Incremental improvement

Balanced accuracy averages hunting recall and no-hunting recall. The paired
ranges resample complete cars 2,000 times from fixed held-out predictions.
They do not include retraining or uncertainty from exploratory feature choices.
A range crossing zero indicates an inconclusive comparison.


In [ ]:
paired = comparisons.set_index("criterion")
comparison_table = (100 * paired[
    ["change_in_balanced_accuracy", "lower_95", "upper_95"]
]).round(1).rename(columns={
    "change_in_balanced_accuracy": "Gain (percentage points)",
    "lower_95": "95% lower", "upper_95": "95% upper",
})
comparison_table["Cars with fewer errors"] = paired.cars_with_fewer_errors
comparison_table["Cars with more errors"] = paired.cars_with_more_errors
comparison_table["Cars tied"] = paired.cars_tied
display(comparison_table)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from profile_shapes import TREAD_GRID

hunting_counts = truck_data.groupby(["Car", "Truck"])["displacement_hunted"].sum()
slope_scale = np.std(TREAD_GRID)
truck_mismatch = {}

for car, truck in hunting_counts.index:
    axles = (1, 2) if truck == 1 else (3, 4)

    # Pick the axle with the largest overall left/right profile difference.
    dominant_axle = max(
        axles,
        key=lambda axle: np.linalg.norm(
            profiles[car, axle, "L"] - profiles[car, axle, "R"]
        ),
    )

    slopes = wheel_data.loc[
        (wheel_data["Car"] == car) & (wheel_data["Axle"] == dominant_axle)
    ].set_index("Side")["tread_slope"]
    truck_mismatch[car, truck] = abs(slopes["L"] - slopes["R"]) * slope_scale

paired = shape_data.set_index(["Car", "Truck"])[["mean_tread_nonlinearity"]].copy()
paired["slope_mismatch"] = pd.Series(truck_mismatch)
paired["hunting_count"] = hunting_counts
delta = paired.xs(2, level="Truck") - paired.xs(1, level="Truck")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
panels = [
    ("mean_tread_nonlinearity", "Tread nonlinearity", {4: (-11, 8), 14: (5, -14)}),
    ("slope_mismatch", "Left/right slope mismatch", {7: (-13, -14), 8: (6, 8)}),
]

for ax, (feature, title, label_offsets) in zip(axes, panels):
    ax.scatter(delta[feature], delta["hunting_count"], color="#226584")
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
    for car, row in delta.iterrows():
        ax.annotate(
            str(car),
            (row[feature], row["hunting_count"]),
            xytext=label_offsets.get(car, (4, 4)),
            textcoords="offset points",
            fontsize=9,
        )
    ax.set(title=title, xlabel="Truck 2 − Truck 1 shape measurement")
    ax.grid(alpha=0.15)

axes[0].set_ylabel("Truck 2 − Truck 1 hunting count")
fig.tight_layout()
delta.to_csv(figure_folder / "within_car_shape_comparison_plot_data.csv")
fig.savefig(figure_folder / "within_car_shape_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig(figure_folder / "within_car_shape_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Compare balanced accuracy for the two hunting criteria.
accuracy_pct = 100 * performance["balanced_accuracy"].unstack("model")
models = ["Mean conicity + rail", "Mean conicity + shape + rail"]
panels = [("Acceleration", "Acceleration"), ("Displacement sensitivity", "Displacement")]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), sharey=True)

for ax, (criterion, title) in zip(axes, panels):
    scores = accuracy_pct.loc[criterion, models]
    bars = ax.bar(
        ["Conicity + rail", "Add three\nshape descriptors"],
        scores,
        color=["#8a969e", "#226584"],
        width=0.6,
    )
    ax.bar_label(bars, fmt="%.1f%%", padding=4)
    ax.set(title=title, ylim=(0, 100))
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

axes[0].set_ylabel("Balanced accuracy (%)")
fig.tight_layout()
performance.to_csv(figure_folder / "profile_shape_performance_plot_data.csv")
fig.savefig(figure_folder / "profile_shape_performance.png", dpi=300, bbox_inches="tight")
fig.savefig(figure_folder / "profile_shape_performance.pdf", bbox_inches="tight")
plt.show()

## Sensitivity to individual cars

I recalculate the score difference after omitting each car's held-out predictions
in turn. This checks whether one car accounts for the gain. It does not refit
models without that car and is not a second validation experiment.


In [ ]:
influence_rows = []
for target, predictions in local_predictions.items():
    for car in sorted(predictions.Car.unique()):
        remaining = predictions.loc[predictions.Car != car]
        base = classification_metrics(remaining.actual, remaining["Mean conicity + rail"])
        shape = classification_metrics(remaining.actual, remaining["Mean conicity + shape + rail"])
        influence_rows.append({
            "criterion": target_names[target], "omitted_car": car,
            "gain": shape["balanced_accuracy"] - base["balanced_accuracy"],
        })

influence = pd.DataFrame(influence_rows)
display((100 * influence.groupby("criterion").gain.agg(["min", "max"]))
        .round(1).rename(columns={"min": "Smallest gain (pp)",
                                 "max": "Largest gain (pp)"}))


## Interpretation

The comparison tests the three descriptors together. A gain does not identify
which measurement explains hunting or establish a cause. These features may
also reflect other differences between cars.

Whole-car splits prevent the same car's wheels and rail assessments from
appearing in training and test together. The number of independent cars remains
small, and repeated exploration of this dataset makes these results provisional.
The displacement comparison stays separate from the main acceleration result.
The experiment does not test new rail types or operating conditions.
